# Full pipeline: residual key → panel → simulation

One notebook, start to finish: a `residual_key` string and a `candidate_panel_path` in, a `SimulationResult` out. It downloads market data if it isn't cached yet, builds and persists the candidate panel, then simulates against it. For the panel-building step in detail, see [01_create_candidate_panel.ipynb](01_create_candidate_panel.ipynb); for result inspection, see [02_run_simulation.ipynb](02_run_simulation.ipynb).

In [1]:
# ── knobs ─────────────────────────────────────────────────────────────
RESIDUAL_KEY = "exp_hl504_mh1008_rf"   # parsed via CausalResidualConfig.from_key
CANDIDATE_PANEL_PATH = "howto3"        # subdir under CANDIDATE_PANELS_ROOT — panel build and
                                        # simulation both read/write here; defined once, reused below
SELECTED_GROUPS = ["materials"]
ENTRY_Z = 1.5
Z_LOOKBACK = 21
Z_METHOD = "ewm"

# panel-build windows — PanelBatchConfig's concern, independent of the z-score above
HEDGE_RATIO_LB = 252
MR_DIAG_LB = 252
MAX_STEPS = 60                         # notebook-speed cap on panel creation (like notebook 01)

# No default (mandatory, absolute, no fraction of HEDGE_RATIO_LB) — a pair
# only gets a hedge ratio if it retained the full requested window.
MIN_OBS = HEDGE_RATIO_LB


## 1. Ensure market data is present (download only if missing)

In [2]:
from src.data.universe_loader import ensure_universe_data

ensure_universe_data(SELECTED_GROUPS)


Loading universe materials_only_v1...
  cache hit — loading materials_only_v1 from disk (/home/nikolajnock/PycharmProjects/statarb_sim/data/market/universes/materials_only_v1/prices_daily.parquet)
QCWarning(ticker='APD', issue='open_outside_range', details='1 rows where Open is outside [Low, High]')
QCWarning(ticker='AVY', issue='open_outside_range', details='1 rows where Open is outside [Low, High]')
QCWarning(ticker='BALL', issue='open_outside_range', details='1 rows where Open is outside [Low, High]')
QCWarning(ticker='CF', issue='open_outside_range', details='1 rows where Open is outside [Low, High]')
QCWarning(ticker='IFF', issue='open_outside_range', details='1 rows where Open is outside [Low, High]')
QCWarning(ticker='IP', issue='open_outside_range', details='1 rows where Open is outside [Low, High]')
QCWarning(ticker='NUE', issue='open_outside_range', details='1 rows where Open is outside [Low, High]')
QCWarning(ticker='SHW', issue='open_outside_range', details='1 rows where Op

## 2. Build and persist the candidate panel

In [3]:
from src.candidates.panel_batch import PanelBatchConfig, run_panel_batch
from src.candidates.pair_candidate_panel_creator import PairSpreadConfig

panel_cfg = PanelBatchConfig(
    residual_configs=[RESIDUAL_KEY],   # list[str] — resolved via CausalResidualConfig.from_key
    hedge_ratio_lb=HEDGE_RATIO_LB,
    mr_diag_lb=MR_DIAG_LB,
    selected_groups=SELECTED_GROUPS,
    frequency="W-FRI",
    max_steps=MAX_STEPS,
    pair_cfg=PairSpreadConfig(
        hedge_ratio_methods=["pca"],
        min_obs=MIN_OBS,
        min_return_std=1e-8,
        min_level_std=1e-8,
        min_kappa=1e-6,
        max_half_life=126.0,
        tiny_weight_threshold=1e-6,
    ),
    persist_result=True,
    persist_residual_params=True,
    persist_dir_template=CANDIDATE_PANEL_PATH,
)
panel_results = run_panel_batch(panel_cfg)
print(f"panels built: {list(panel_results.keys())}")


Batch 20260908_1241
Groups: 1 [mat]
Residual configs: ['exp_hl504_mh1008_rf']
Total jobs: 1


Groups:   0%|          | 0/1 [00:00<?, ?group/s]

Loading universe materials_only_v1...
  cache hit — loading materials_only_v1 from disk (/home/nikolajnock/PycharmProjects/statarb_sim/data/market/universes/materials_only_v1/prices_daily.parquet)


  mat / exp_hl504_mh1008_rf: 4366/4446 valid candidates

Done. 1 panels created.
panels built: [('materials', 'exp_hl504_mh1008_rf')]


## 3. Run the simulation against the panel just built

`SweepConfig` + `run_sweep` is the minimal simulator entry point. Even with a single
residual timescale, `RESIDUAL_KEY` must be threaded through `z_score_overrides` (not the
plain `z_lookback`/`z_method` fields) — the panel just built carries its real,
non-empty `residual_key`, and the simulator matches `ZScoreConfig.residual_key` against
it to select the panel. `start_date`/`end_date` are left unset so the run covers exactly
the (small, capped) date range the panel above was built over.

In [4]:
from src.simulator.config import ZScoreConfig
from src.simulator.sweep_runner import SweepConfig, run_sweep

RUNS = [
    SweepConfig(
        entry_z=ENTRY_Z,
        z_score_overrides=[ZScoreConfig(lookback=Z_LOOKBACK, ddof=1, method=Z_METHOD, residual_key=RESIDUAL_KEY)],
        candidate_panel_subdir=CANDIDATE_PANEL_PATH,
        start_date=None,
        end_date=None,
    ),
]
# skip_existing=False: this howto should always produce a fresh result to inspect below,
# even if an identical config is already sitting in sweep_results.pkl from a prior run.
sweep_df, sweep_results = run_sweep(RUNS, skip_existing=False)
result = sweep_results[0]


[sweep 1/1] z1.5_xz0.0_mt1_ewm_rhl504_zhl21_bpn100k_voln_tick0.15_gx10.0_all [67054c29]
[run] Loading universe market data...
[discover] Found 1 groups: ['mat']
[factory] Loading 1 universe(s)...
Loading universe materials_only_v1...
  cache hit — loading materials_only_v1 from disk (/home/nikolajnock/PycharmProjects/statarb_sim/data/market/universes/materials_only_v1/prices_daily.parquet)
QCWarning(ticker='APD', issue='open_outside_range', details='1 rows where Open is outside [Low, High]')
QCWarning(ticker='AVY', issue='open_outside_range', details='1 rows where Open is outside [Low, High]')
QCWarning(ticker='BALL', issue='open_outside_range', details='1 rows where Open is outside [Low, High]')
QCWarning(ticker='CF', issue='open_outside_range', details='1 rows where Open is outside [Low, High]')
QCWarning(ticker='IFF', issue='open_outside_range', details='1 rows where Open is outside [Low, High]')
QCWarning(ticker='IP', issue='open_outside_range', details='1 rows where Open is outsid

Simulating:   0%|          | 0/286 [00:00<?, ?step/s]

[signals] 78/78 spread level series missing under /home/nikolajnock/PycharmProjects/statarb_sim/artifacts/candidate_panels/howto3 on 2009-08-14 (e.g. IFF|IP); recomputing from residuals.


[signals] 78/78 spread level series missing under /home/nikolajnock/PycharmProjects/statarb_sim/artifacts/candidate_panels/howto3 on 2009-08-17 (e.g. IFF|IP); recomputing from residuals.


[signals] 78/78 spread level series missing under /home/nikolajnock/PycharmProjects/statarb_sim/artifacts/candidate_panels/howto3 on 2009-08-18 (e.g. IFF|IP); recomputing from residuals.


...

[signals] 82/82 spread level series missing under /home/nikolajnock/PycharmProjects/statarb_sim/artifacts/candidate_panels/howto3 on 2010-10-01 (e.g. NEM|NUE); recomputing from residuals.


Simulating: 100%|██████████| 286/286 [00:08<00:00, 35.16step/s, date=2010-10-01]

[sim] Year 2010 checkpoint: flushed logs, memory freed



────────────────────────────────────────────────────
  Strategy Performance
────────────────────────────────────────────────────
  Total Return (Net)                          9.33%
  Total Return (Gross)                        9.99%
  Annual Return (Net)                         8.20%
  Annual Return (Gross)                       8.78%
  Sharpe Ratio (Net)                          0.514
  Sortino Ratio (Net)                         0.550
  Sortino Ratio (Gross)                       0.582
  Calmar Ratio (Net)                          0.468
  Max Drawdown (Net)                        -17.52%
  Max Drawdown (Gross)                      -17.34%
  # Trades                                      133
  # Groups Traded                                 1
  # Trading Days                                285
  Avg Concurrent Positions                    20.48
  Avg Holding Period (days)                    38.0
  Avg Holding — Wins (days)                    31.6
  Avg Holding — Losses (days)         


[performance] HTML report written to: /home/nikolajnock/PycharmProjects/statarb_sim/artifacts/simulation_runs/20260908_1241_67054c29/strategy_performance_quantstats.html
[sweep] Saved 15 results (0 skipped)


## 4. Result summary

In [5]:
print(f"closed trades: {len(result.closed_trades)}")

m = result.performance.metrics
for k in ["n_trades", "win_rate_net", "sharpe_net", "total_net_pnl", "max_drawdown_net"]:
    if k in m:
        print(f"  {k:18s}: {m[k]}")


closed trades: 133
  n_trades          : 133
  win_rate_net      : 0.6691729323308271
  sharpe_net        : 0.5137567387662696
  total_net_pnl     : 211663.1381058168
  max_drawdown_net  : -0.17517101368236604
